In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent.parent))

from  lora_transfer_pruning.core.pruning_instrumentor import PruningInstrumentor
from transformers import AutoModelForCausalLM
import torch
import compare_utils
from compare_utils import debug_group_prune_step_by_step
from lora_transfer_pruning.adapter.torch_pruning.torch_pruning_group_builder import TorchPruningGroupBuilder
from compare_utils import full_attention_test_with_prune
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch.nn as nn

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


In [2]:
MODEL = "deepseek-ai/DeepSeek-V2-Lite-Chat" 
DTYPE = torch.float16
DEVICE = "cuda:3"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=DTYPE,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    llm_int8_skip_modules=["lm_head", "mlp.gate"],
)

tokenizer = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=False)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

def load_model():
    model = AutoModelForCausalLM.from_pretrained(
        MODEL, trust_remote_code=False, device_map={"": DEVICE}, dtype=DTYPE,
        #quantization_config=quantization_config, 
        attn_implementation="eager",
    )
    model.eval()
    router = model.model.layers[1].mlp.gate
    assert isinstance(router, nn.Linear), type(router)
    assert tuple(router.weight.shape) == (model.config.n_routed_experts, model.config.hidden_size)
    return model

In [3]:
model = load_model()

Loading weights:   0%|          | 0/351 [00:00<?, ?it/s]

In [4]:
model

DeepseekV2ForCausalLM(
  (model): DeepseekV2Model(
    (embed_tokens): Embedding(102400, 2048)
    (layers): ModuleList(
      (0): DeepseekV2DecoderLayer(
        (self_attn): DeepseekV2Attention(
          (q_proj): Linear(in_features=2048, out_features=3072, bias=False)
          (kv_a_proj_with_mqa): Linear(in_features=2048, out_features=576, bias=False)
          (kv_a_layernorm): DeepseekV2RMSNorm((512,), eps=1e-06)
          (kv_b_proj): Linear(in_features=512, out_features=4096, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): DeepseekV2MLP(
          (gate_proj): Linear(in_features=2048, out_features=10944, bias=False)
          (up_proj): Linear(in_features=2048, out_features=10944, bias=False)
          (down_proj): Linear(in_features=10944, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): DeepseekV2RMSNorm((2048,), eps=1e-06)
        (post_attention_layern

In [5]:
from transformer_lens.model_bridge import TransformerBridge
import transformer_lens

bridge = TransformerBridge.boot_transformers(
    MODEL,
    hf_model=model,
    dtype=torch.float16,
)

In [6]:
bridge


TransformerBridge(
  (embed): EmbeddingBridge(
    (hook_in): HookPoint(name='embed.hook_in')
    (hook_out): HookPoint(name='embed.hook_out')
    (_original_component): Embedding(102400, 2048)
  )
  (rotary_emb): RotaryEmbeddingBridge(
    (hook_in): HookPoint(name='rotary_emb.hook_in')
    (hook_out): HookPoint(name='rotary_emb.hook_out')
    (hook_cos): HookPoint(name='rotary_emb.hook_cos')
    (hook_sin): HookPoint(name='rotary_emb.hook_sin')
    (_original_component): DeepseekV2RotaryEmbedding()
  )
  (blocks): ModuleList(
    (0): MLABlockBridge(
      (hook_in): HookPoint(name='blocks.0.hook_in')
      (hook_out): HookPoint(name='blocks.0.hook_out')
      (hook_mlp_in): HookPoint(name='blocks.0.hook_mlp_in')
      (_original_component): DeepseekV2DecoderLayer(
        (self_attn): MLAAttentionBridge(
          (hook_in): HookPoint(name='blocks.0.attn.hook_in')
          (hook_out): HookPoint(name='blocks.0.attn.hook_out')
          (hook_attn_scores): HookPoint(name='blocks.0.at

In [7]:
# model.model.language_model.layers[0].self_attn.is_kv_shared_layer

In [8]:
# model.model.language_model.layers[0].self_attn.kv_shared_layer_index

In [9]:
# model.model.language_model.layers[0].self_attn.store_full_length_kv

In [10]:
# model.model.layers[0].self_attn.config.use_alternative_attention

In [11]:
model.model.layers[0].self_attn.o_proj._original_component.bias

In [12]:
model.model.layers[0].self_attn._original_component.config.attention_bias

False

In [13]:
model.model.layers[0].self_attn._original_component.config._attn_implementation

'eager'

In [14]:
# model.model.layers[0].self_attn._original_component.config.use_double_wide_mlp

In [15]:
from transformers import DeepseekV2ForCausalLM

In [16]:
bridge.blocks[0].mlp #ordinary mlp
#but its only share expert, not DeepseekV2Experts

MoEBridge(
  (hook_in): HookPoint(name='blocks.0.mlp.hook_in')
  (hook_out): HookPoint(name='blocks.0.mlp.hook_out')
  (hook_router_scores): HookPoint(name='blocks.0.mlp.hook_router_scores')
  (_original_component): DeepseekV2MLP(
    (gate_proj): Linear(in_features=2048, out_features=10944, bias=False)
    (up_proj): Linear(in_features=2048, out_features=10944, bias=False)
    (down_proj): Linear(in_features=10944, out_features=2048, bias=False)
    (act_fn): SiLUActivation()
  )
)

In [17]:
#MOE also implemented via nn.linear inside forward
#we have only shared mlp for changing with hooks.

In [18]:
bridge.blocks[0]._original_component.mlp._original_component

DeepseekV2MLP(
  (gate_proj): Linear(in_features=2048, out_features=10944, bias=False)
  (up_proj): Linear(in_features=2048, out_features=10944, bias=False)
  (down_proj): Linear(in_features=10944, out_features=2048, bias=False)
  (act_fn): SiLUActivation()
)

In [19]:
bridge.blocks[0]

MLABlockBridge(
  (hook_in): HookPoint(name='blocks.0.hook_in')
  (hook_out): HookPoint(name='blocks.0.hook_out')
  (hook_mlp_in): HookPoint(name='blocks.0.hook_mlp_in')
  (_original_component): DeepseekV2DecoderLayer(
    (self_attn): MLAAttentionBridge(
      (hook_in): HookPoint(name='blocks.0.attn.hook_in')
      (hook_out): HookPoint(name='blocks.0.attn.hook_out')
      (hook_attn_scores): HookPoint(name='blocks.0.attn.hook_attn_scores')
      (hook_pattern): HookPoint(name='blocks.0.attn.hook_pattern')
      (hook_hidden_states): HookPoint(name='blocks.0.attn.hook_hidden_states')
      (hook_result): HookPoint(name='blocks.0.attn.hook_result')
      (hook_rot_k): HookPoint(name='blocks.0.attn.hook_rot_k')
      (hook_rot_q): HookPoint(name='blocks.0.attn.hook_rot_q')
      (hook_cos): HookPoint(name='blocks.0.attn.hook_cos')
      (hook_sin): HookPoint(name='blocks.0.attn.hook_sin')
      (hook_q_latent): HookPoint(name='blocks.0.attn.hook_q_latent')
      (hook_kv_latent): HookP

In [20]:
# model.model.layers[0]._original_component.hidden_size_per_layer_input

In [21]:
model.model.layers[0]._original_component.hidden_size

2048

In [22]:
# model.model.layers[0]._original_component.config.hidden_activation

In [23]:
from datasets import load_dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL)
validation_dataset = load_dataset(
    "Salesforce/wikitext",
    "wikitext-2-raw-v1",
    split="validation",
)
validation_dataset

Dataset({
    features: ['text'],
    num_rows: 3760
})

In [24]:
for i, text in enumerate(validation_dataset):
    print(f"{i}: {text}")
    if i > 10:
        break

0: {'text': ''}
1: {'text': ' = Homarus gammarus = \n'}
2: {'text': ''}
3: {'text': ' Homarus gammarus , known as the European lobster or common lobster , is a species of clawed lobster from the eastern Atlantic Ocean , Mediterranean Sea and parts of the Black Sea . It is closely related to the American lobster , H. americanus . It may grow to a length of 60 cm ( 24 in ) and a mass of 6 kilograms ( 13 lb ) , and bears a conspicuous pair of claws . In life , the lobsters are blue , only becoming " lobster red " on cooking . Mating occurs in the summer , producing eggs which are carried by the females for up to a year before hatching into planktonic larvae . Homarus gammarus is a highly esteemed food , and is widely caught using lobster pots , mostly around the British Isles . \n'}
4: {'text': ''}
5: {'text': ' = = Description = = \n'}
6: {'text': ''}
7: {'text': ' Homarus gammarus is a large crustacean , with a body length up to 60 centimetres ( 24 in ) and weighing up to 5 – 6 kilogram

In [25]:
CONTEXT_LENGTH = 256
NUM_EVAL_BLOCKS = 32 #(block=batch)
EVAL_BATCH_SIZE = 1

validation_text = "\n\n".join(
    text for text in validation_dataset["text"] if text.strip()
)
validation_tokens = tokenizer(
    validation_text,
    add_special_tokens=False,
    return_tensors="pt",
).input_ids[0]

num_blocks = NUM_EVAL_BLOCKS
assert num_blocks > 0, "Validation split does not contain enough tokens"
evaluation_blocks = validation_tokens[: num_blocks * CONTEXT_LENGTH].reshape(
    num_blocks, CONTEXT_LENGTH
)
evaluation_blocks.shape

torch.Size([32, 256])

In [26]:
ignored_params = []
# for name, param in model.named_parameters():
#     if "norm" in name:
#         ignored_params.append(param)

In [27]:
import torch
import torch.nn as nn
import torch_pruning as tp

example_inputs = evaluation_blocks[:4].to(DEVICE)

def trace_forward(model, input_ids):
    return model(
        input=input_ids,
        use_cache=False,
        return_type="logits",
        #return_dict=True,
    ) #for compatability with tp

DG = tp.DependencyGraph().build_dependency(
    bridge,
    example_inputs=example_inputs,
    forward_fn=trace_forward,
    ignored_params=ignored_params,
        unwrapped_parameters=[]
)


/glazkov-dev/LoRa-Transfer-Pruning/.venv/lib/python3.10/site-packages/torch_pruning/dependency/graph.py:390: UserWarning: Unwrapped parameters detected: ['model.layers.2._original_component.input_layernorm._original_component.weight', 'model.layers.24._original_component.mlp._original_component.experts.gate_up_proj', 'model.layers.4._original_component.post_attention_layernorm._original_component.weight', 'model.layers.12._original_component.post_attention_layernorm._original_component.weight', 'model.layers.20._original_component.mlp._original_component.experts.gate_up_proj', 'model.layers.24._original_component.input_layernorm._original_component.weight', 'model.layers.13._original_component.mlp._original_component.experts.down_proj', 'model.layers.24._original_component.self_attn._original_component.kv_a_layernorm._original_component.weight', 'model.layers.25._original_component.mlp._original_component.experts.gate_up_proj', 'model.layers.2._original_component.mlp._original_componen

In [28]:
bridge.blocks[0].attn

MLAAttentionBridge(
  (hook_in): HookPoint(name='blocks.0.attn.hook_in')
  (hook_out): HookPoint(name='blocks.0.attn.hook_out')
  (hook_attn_scores): HookPoint(name='blocks.0.attn.hook_attn_scores')
  (hook_pattern): HookPoint(name='blocks.0.attn.hook_pattern')
  (hook_hidden_states): HookPoint(name='blocks.0.attn.hook_hidden_states')
  (hook_result): HookPoint(name='blocks.0.attn.hook_result')
  (hook_rot_k): HookPoint(name='blocks.0.attn.hook_rot_k')
  (hook_rot_q): HookPoint(name='blocks.0.attn.hook_rot_q')
  (hook_cos): HookPoint(name='blocks.0.attn.hook_cos')
  (hook_sin): HookPoint(name='blocks.0.attn.hook_sin')
  (hook_q_latent): HookPoint(name='blocks.0.attn.hook_q_latent')
  (hook_kv_latent): HookPoint(name='blocks.0.attn.hook_kv_latent')
  (hook_q): HookPoint(name='blocks.0.attn.hook_q')
  (hook_k): HookPoint(name='blocks.0.attn.hook_k')
  (hook_v): HookPoint(name='blocks.0.attn.hook_v')
  (_original_component): DeepseekV2Attention(
    (q_proj): LinearBridge(2048 -> 3072, bi

In [29]:
hasattr(bridge.blocks[0].attn, 'q') #no standart kqvo!

False

In [30]:
bridge.blocks[0].attn.kv_lora_rank

512

In [31]:
bridge.blocks[0].attn.q_proj

LinearBridge(2048 -> 3072, bias=False, original_component=Linear)

In [32]:
bridge.blocks[0].attn.kv_a_proj_with_mqa

LinearBridge(2048 -> 576, bias=False, original_component=Linear)

In [33]:
bridge.blocks[0].attn.kv_b_proj

LinearBridge(512 -> 4096, bias=False, original_component=Linear)

In [34]:
bridge.blocks[0].attn.o_proj

LinearBridge(2048 -> 2048, bias=False, original_component=Linear)

In [35]:
bridge.blocks[0].attn.q_lora_rank #None - so without q_a_proj and other

In [36]:
hasattr(bridge, 'rotary_emb')

True

In [37]:
bridge.get_submodule("blocks.0.attn.hook_q")

HookPoint(name='blocks.0.attn.hook_q')

In [38]:
# # hidden_states
# │
# ├── Q path
# │   ├── q_a_proj if lora, else q_proj
# │   ├── q_a_layernorm
# │   ├── q_b_proj
# │   ├── split → q_nope, q_pe
# │   ├── RoPE(q_pe)
# │   └── cat(q_nope, q_pe) → Q
# │
# └── KV path
#     ├── kv_a_proj_with_mqa
#     ├── split
#     │   ├── kv_latent
#     │   │   ├── kv_a_layernorm
#     │   │   ├── kv_b_proj
#     │   │   └── split → k_nope, V
#     │   │
#     │   └── k_pe
#     │       ├── RoPE
#     │       └── expand на все головы
#     │
#     └── cat(k_nope, k_pe) → K

# attention(Q, K, V)
#     ↓
# reshape
#     ↓
# o_proj

In [39]:
print(bridge.blocks[0].attn._original_component.kv_lora_rank)
print(bridge.blocks[0].attn._original_component.qk_nope_head_dim)
print(bridge.blocks[0].attn._original_component.qk_rope_head_dim)
print(bridge.blocks[0].attn._original_component.v_head_dim)

512
128
64
128


In [40]:
#name of module, cols (in), rows(out)


#local configuration
d = {"blocks.0.attn.q": (None, [2, 6, 9]), #repeat indices for qkvo
     "blocks.0.mlp.up_proj": (None, [1, 3, 5])}

In [41]:
bridge.blocks[0].attn.q_proj.hook_in.name

'blocks.0.attn.q_proj.hook_in'

In [42]:
from transformers import DeepseekV2Model


isinstance(bridge.model, DeepseekV2Model)

True

In [43]:
bridge.blocks[0].attn.kv_lora_rank

512

In [44]:
group = DG.get_pruning_group(
    bridge.blocks[0].attn.q_proj._original_component, 
    tp.prune_linear_out_channels, 
    idxs=[2, 6, 9] )

In [45]:
print(group)


--------------------------------
          Pruning Group
--------------------------------
[0] prune_out_channels on blocks.0._original_component.self_attn._original_component.q_proj._original_component (Linear(in_features=2048, out_features=3072, bias=False)) => prune_out_channels on blocks.0._original_component.self_attn._original_component.q_proj._original_component (Linear(in_features=2048, out_features=3072, bias=False)), len(idxs)=3
[1] prune_out_channels on blocks.0._original_component.self_attn._original_component.q_proj._original_component (Linear(in_features=2048, out_features=3072, bias=False)) => prune_out_channels on _Reshape_1608(), len(idxs)=3
[2] prune_out_channels on _Reshape_1608() => prune_out_channels on _ElementWiseOp_1607(TransposeBackward0), len(idxs)=3
[3] prune_out_channels on _ElementWiseOp_1607(TransposeBackward0) => prune_out_channels on _SplitOp_1599(None), len(idxs)=3
[4] prune_out_channels on _SplitOp_1599(None) => prune_out_channels on _ConcatOp_1598(Non

In [46]:
group = DG.get_pruning_group(
    bridge.blocks[0].attn.kv_a_proj_with_mqa._original_component, 
    tp.prune_linear_out_channels,   
    idxs=[2, 6, 9] )

In [47]:
print(group)


--------------------------------
          Pruning Group
--------------------------------
[0] prune_out_channels on blocks.0._original_component.self_attn._original_component.kv_a_proj_with_mqa._original_component (Linear(in_features=2048, out_features=576, bias=False)) => prune_out_channels on blocks.0._original_component.self_attn._original_component.kv_a_proj_with_mqa._original_component (Linear(in_features=2048, out_features=576, bias=False)), len(idxs)=3
[1] prune_out_channels on blocks.0._original_component.self_attn._original_component.kv_a_proj_with_mqa._original_component (Linear(in_features=2048, out_features=576, bias=False)) => prune_out_channels on _SplitOp_1564(None), len(idxs)=3
[2] prune_out_channels on _SplitOp_1564(None) => prune_out_channels on _ElementWiseOp_1559(ToCopyBackward0), len(idxs)=3
[3] prune_out_channels on _SplitOp_1564(None) => prune_out_channels on _Reshape_1596(), len(idxs)=3
[4] prune_out_channels on _Reshape_1596() => prune_out_channels on _Element

In [48]:
def manually_indices_repeating(num_heads: int, head_dim: int, pruning_indices: torch.Tensor):
    all_indices = []
    for head_num in range(num_heads):
        all_indices.append(
            pruning_indices+head_num*head_dim)
    return torch.cat(all_indices)

In [49]:
type(bridge.blocks[0].attn)

transformer_lens.model_bridge.generalized_components.mla_attention.MLAAttentionBridge

In [50]:
bridge

TransformerBridge(
  (embed): EmbeddingBridge(
    (hook_in): HookPoint(name='embed.hook_in')
    (hook_out): HookPoint(name='embed.hook_out')
    (_original_component): Embedding(102400, 2048)
  )
  (rotary_emb): RotaryEmbeddingBridge(
    (hook_in): HookPoint(name='rotary_emb.hook_in')
    (hook_out): HookPoint(name='rotary_emb.hook_out')
    (hook_cos): HookPoint(name='rotary_emb.hook_cos')
    (hook_sin): HookPoint(name='rotary_emb.hook_sin')
    (_original_component): DeepseekV2RotaryEmbedding()
  )
  (blocks): ModuleList(
    (0): MLABlockBridge(
      (hook_in): HookPoint(name='blocks.0.hook_in')
      (hook_out): HookPoint(name='blocks.0.hook_out')
      (hook_mlp_in): HookPoint(name='blocks.0.hook_mlp_in')
      (_original_component): DeepseekV2DecoderLayer(
        (self_attn): MLAAttentionBridge(
          (hook_in): HookPoint(name='blocks.0.attn.hook_in')
          (hook_out): HookPoint(name='blocks.0.attn.hook_out')
          (hook_attn_scores): HookPoint(name='blocks.0.at

In [51]:
#matches with config from llama
print(bridge.blocks[0].attn._original_component.config.num_attention_heads)
print(bridge.blocks[0].attn._original_component.config.head_dim)
print(bridge.blocks[0].attn._original_component.config.num_key_value_heads)

16
64
16


In [52]:
print(bridge.blocks[0].attn.q_proj._original_component.weight.shape)
print(bridge.blocks[0].attn.kv_b_proj._original_component.weight.shape)
print("no informative...")

torch.Size([3072, 2048])
torch.Size([4096, 512])
no informative...


In [53]:
bridge.blocks[0].attn._original_component.config.hidden_size

2048

In [54]:
bridge.blocks[0].attn.head_dim

64

In [55]:
repeated_idxs = manually_indices_repeating(
    bridge.blocks[0].attn._original_component.config.num_attention_heads,
    bridge.blocks[0].attn._original_component.config.head_dim,
    torch.tensor([2, 6, 9])
)
repeated_idxs

tensor([  2,   6,   9,  66,  70,  73, 130, 134, 137, 194, 198, 201, 258, 262,
        265, 322, 326, 329, 386, 390, 393, 450, 454, 457, 514, 518, 521, 578,
        582, 585, 642, 646, 649, 706, 710, 713, 770, 774, 777, 834, 838, 841,
        898, 902, 905, 962, 966, 969])

In [56]:
# for i, (dep, idx) in enumerate(group):
#     if (isinstance(dep.target.module, nn.Parameter)):
#         print(dep.target.name)
#         print("q norm", dep.target.module is bridge.blocks[0].attn.q_norm.weight)
#         print("k norm", dep.target.module is bridge.blocks[0].attn.k_norm.weight)
#         print(dep.handler)
#         print(idx)
#         print("param to name", group._DG._param_to_name[dep.target.module])
        # print(dep.target.module)

In [57]:
from transformer_lens.model_bridge.generalized_components.base import GeneralizedComponent


for name, mod in bridge.named_modules():
    if (isinstance(mod, TransformerBridge)):
        continue
    have = False
    if ("rms" in mod.__class__.__name__.lower()):
        have = True
    comp = False
    if (isinstance(mod, GeneralizedComponent)):
        comp = True
    print(name, have, comp)

embed False True
embed.hook_in False False
embed.hook_out False False
embed._original_component False False
rotary_emb False True
rotary_emb.hook_in False False
rotary_emb.hook_out False False
rotary_emb.hook_cos False False
rotary_emb.hook_sin False False
rotary_emb._original_component False False
blocks False False
blocks.0 False True
blocks.0.hook_in False False
blocks.0.hook_out False False
blocks.0.hook_mlp_in False False
blocks.0._original_component False False
blocks.0._original_component.self_attn False True
blocks.0._original_component.self_attn.hook_in False False
blocks.0._original_component.self_attn.hook_out False False
blocks.0._original_component.self_attn.hook_attn_scores False False
blocks.0._original_component.self_attn.hook_pattern False False
blocks.0._original_component.self_attn.hook_hidden_states False False
blocks.0._original_component.self_attn.hook_result False False
blocks.0._original_component.self_attn.hook_rot_k False False
blocks.0._original_component.sel

Why only q_norm and k_norm found? Because v_norm without .weight.

So unwrapped RMS norms founded and added to deps in group automatically.

### Without MOE block, just mlp

In [58]:
from transformers.models.deepseek_v2.modeling_deepseek_v2 import DeepseekV2MLP
group = DG.get_pruning_group(
    bridge.blocks[0].mlp.up_proj, 
    tp.prune_linear_out_channels, 
    idxs=repeated_idxs.tolist() )

In [59]:
print(group)
print("group find fine!")


--------------------------------
          Pruning Group
--------------------------------
[0] prune_out_channels on blocks.0._original_component.mlp._original_component.up_proj (Linear(in_features=2048, out_features=10944, bias=False)) => prune_out_channels on blocks.0._original_component.mlp._original_component.up_proj (Linear(in_features=2048, out_features=10944, bias=False)), len(idxs)=48
[1] prune_out_channels on blocks.0._original_component.mlp._original_component.up_proj (Linear(in_features=2048, out_features=10944, bias=False)) => prune_out_channels on _ElementWiseOp_1525(MulBackward0), len(idxs)=48
[2] prune_out_channels on _ElementWiseOp_1525(MulBackward0) => prune_out_channels on _ElementWiseOp_1526(SiluBackward0), len(idxs)=48
[3] prune_out_channels on _ElementWiseOp_1525(MulBackward0) => prune_out_channels on _Reshape_1523(), len(idxs)=48
[4] prune_out_channels on _Reshape_1523() => prune_out_channels on _ElementWiseOp_1522(MmBackward0), len(idxs)=48
[5] prune_out_channels

In [60]:
bridge.blocks[0].mlp._original_component.act_fn

SiLUActivation()

### Fraction-based pruning comparison

Restart the kernel and run the model/dataset setup cells, but skip the preceding explicit-index pruning cell. This experiment samples channel indices from fractions, records the actual independently sampled RoPE coordinates for each attention layer, then compares activation and structural pruning using the same groups.

The solution is next: idxs = idxs for q_proj of size < qk_head_dim = nope_dim + rope_dim  
The indices for v_proj infers from nope_dim of q_proj  
The indices for kv_lora_rank - optional via kv_lora_idxs

In [61]:
import importlib
importlib.reload(compare_utils)

<module 'compare_utils' from '/glazkov-dev/LoRa-Transfer-Pruning/experiments/compare_tp_and_our/compare_utils.py'>

In [62]:
# Fraction-based version of the activation-vs-structural comparison.
# Run on a freshly loaded, unpruned bridge; skip the preceding explicit-index
# comparison cell after restarting the kernel.
from compare_utils import compare_tp_and_transfer_pruning
from compare_utils import create_prune_task

FRACTION_ATTN_LAYERS = [0, 8, 16]
FRACTION_MLP_LAYERS = [] #4, 12, 20] TODO for deepseek mlp only on layer 0, and other - mlp with shared experts 
#ATTN_OUT_FRACTION = 0.1
#MLP_OUT_FRACTION = 0.3 
ATTN_OUT_FRACTION = [1, 2, 99] #q proj
MLP_OUT_FRACTION = [3, 5, 1000] #up proj
FRACTION_SEED = 0

prune_task = create_prune_task(FRACTION_ATTN_LAYERS, 
                               FRACTION_MLP_LAYERS, 
                               ATTN_OUT_FRACTION, 
                               MLP_OUT_FRACTION,
                               q_proj_name="q_proj", #changed for deepseek
                               mlp_up_proj_name="up_proj",
                               )

In [63]:
bridge.get_submodule("blocks.0.mlp.up_proj")

Linear(in_features=2048, out_features=10944, bias=False)

In [64]:
bridge.config.first_k_dense_replace
# bridge.blocks[1].mlp.shared_experts

1

In [65]:
bridge.blocks[0].mlp._original_component.up_proj

Linear(in_features=2048, out_features=10944, bias=False)

Change prune_task to more complex ModelPruneTask with GroupPruneTask to support additional params like `kv_lora_idxs_deepseek`

In [66]:

from lora_transfer_pruning.usecase.local_pruning import LocalPruning

fraction_local_pruning = LocalPruning(
        bridge,
        evaluation_blocks[:1].to(DEVICE),
    )
fraction_groups, _ = fraction_local_pruning.get_torch_pruning_groups_and_structural_setups(
    prune_task
)
mlp_0_group =fraction_local_pruning.torch_pruning_group_builder.DG.get_pruning_group(
            bridge.blocks[0].mlp.up_proj, #костыль, так как up_proj не как bridge хранится. TODO
            tp.prune_linear_out_channels, 
            idxs=MLP_OUT_FRACTION
        )
fraction_groups.append(mlp_0_group)

module=blocks.0.attn.q_proj DeepSeek indices from group:
  q_local=[1, 2, 99]
  q_nope=[1, 2, 99]
  q_rope=[]
  kv_a.out=[]
  kv_b.out=[1, 2, 99, 257, 258, 355, 513, 514, 611, 769, 770, 867, 1025, 1026, 1123, 1281, 1282, 1379, 1537, 1538, 1635, 1793, 1794, 1891, 2049, 2050, 2147, 2305, 2306, 2403, 2561, 2562, 2659, 2817, 2818, 2915, 3073, 3074, 3171, 3329, 3330, 3427, 3585, 3586, 3683, 3841, 3842, 3939]
  kv_b.in=[]
  o_proj.in=[]
module=blocks.8.attn.q_proj DeepSeek indices from group:
  q_local=[1, 2, 99]
  q_nope=[1, 2, 99]
  q_rope=[]
  kv_a.out=[]
  kv_b.out=[1, 2, 99, 257, 258, 355, 513, 514, 611, 769, 770, 867, 1025, 1026, 1123, 1281, 1282, 1379, 1537, 1538, 1635, 1793, 1794, 1891, 2049, 2050, 2147, 2305, 2306, 2403, 2561, 2562, 2659, 2817, 2818, 2915, 3073, 3074, 3171, 3329, 3330, 3427, 3585, 3586, 3683, 3841, 3842, 3939]
  kv_b.in=[]
  o_proj.in=[]
module=blocks.16.attn.q_proj DeepSeek indices from group:
  q_local=[1, 2, 99]
  q_nope=[1, 2, 99]
  q_rope=[]
  kv_a.out=[]
  kv_

So we see a little divergence in more fraction sizes. But it very close.

In [67]:
bridge.blocks.__len__()

27

In [68]:
bridge.blocks[0].attn._original_component.q_proj

LinearBridge(2048 -> 3072, bias=False, original_component=Linear)

In [69]:
#problems with bnbytes linear4bit prune
from bitsandbytes.nn import Linear4bit, Params4bit

for i, (dep, idxs) in enumerate(group):
    module = dep.target.module

    if isinstance(module, Linear4bit):
        print(
            i,
            dep.handler.__name__,
            dep.target.name,
            "logical:",
            module.in_features,
            module.out_features,
            "weight type:",
            type(module.weight),
            "storage shape:",
            tuple(module.weight.shape),
            "dtype:",
            module.weight.dtype,
            "requires_grad:",
            module.weight.requires_grad,
            "bnb_quantized:",
            module.weight.bnb_quantized,
        )

In [70]:
#NOTE: BasePruningFunc._prune_parameter_and_grad - create new Parameter(index_select(old_weight, idxs)), 
#and by default with req_grad=True, that is
#forbidden for uint for Linear4bit

In [71]:
bridge.blocks[0].mlp.up_proj.weight.requires_grad

True

In [72]:
# group.prune()

In [63]:
compare_tp_and_transfer_pruning(bridge,
                                prune_task, #without mlp here TODO
                                FRACTION_SEED,
                                evaluation_blocks,
                                EVAL_BATCH_SIZE
                                )

fraction prune_task:
  blocks.0.attn.q_proj: GroupPruneTask(cols=None, rows=[1, 2, 99], kv_lora_idxs_deepseek=None)
  blocks.8.attn.q_proj: GroupPruneTask(cols=None, rows=[1, 2, 99], kv_lora_idxs_deepseek=None)
  blocks.16.attn.q_proj: GroupPruneTask(cols=None, rows=[1, 2, 99], kv_lora_idxs_deepseek=None)
module=blocks.0.attn.q_proj DeepSeek indices from group:
  q_local=[1, 2, 99]
  q_nope=[1, 2, 99]
  q_rope=[]
  kv_a.out=[]
  kv_b.out=[1, 2, 99, 257, 258, 355, 513, 514, 611, 769, 770, 867, 1025, 1026, 1123, 1281, 1282, 1379, 1537, 1538, 1635, 1793, 1794, 1891, 2049, 2050, 2147, 2305, 2306, 2403, 2561, 2562, 2659, 2817, 2818, 2915, 3073, 3074, 3171, 3329, 3330, 3427, 3585, 3586, 3683, 3841, 3842, 3939]
  kv_b.in=[]
  o_proj.in=[]
module=blocks.8.attn.q_proj DeepSeek indices from group:
  q_local=[1, 2, 99]
  q_nope=[1, 2, 99]
  q_rope=[]
  kv_a.out=[]
  kv_b.out=[1, 2, 99, 257, 258, 355, 513, 514, 611, 769, 770, 867, 1025, 1026, 1123, 1281, 1282, 1379, 1537, 1538, 1635, 1793, 1794, 1

{'baseline': {'loss': 2.390625, 'perplexity': 10.920316696166992},
 'transfer_activation': {'loss': 2.389984130859375,
  'perplexity': 10.913320541381836},
 'torch_pruning_structural': {'loss': 2.390045166015625,
  'perplexity': 10.913987159729004},
 'structural_minus_transfer': {'loss': 6.103515625e-05,
  'perplexity': 0.0006666183471679688}}

Very close!

In [64]:
bridge.blocks[0].attn.qk_nope_head_dim

125

In [65]:
bridge.blocks[0].attn.qk_head_dim

189

In [66]:
bridge.blocks[0].attn.kv_lora_rank

512

In [67]:
bridge.blocks[0].attn.qk_rope_head_dim

64

In [67]:
bridge.blocks[0].attn.kv_a_layernorm._original_component#.weight

DeepseekV2RMSNorm((512,), eps=1e-06)

In [63]:
gb = TorchPruningGroupBuilder(bridge, evaluation_blocks[:1].to(DEVICE))

In [64]:
from lora_transfer_pruning.core.prune_task_type import GroupPruneTask

correct_group, _ = gb.get_correct_pruning_group_and_structural_setup( bridge.blocks[0].attn.q_proj, 
    tp.prune_linear_out_channels,
    GroupPruneTask(None, [2, 3, 5, 7]))

module=blocks.0.attn.q_proj DeepSeek indices from group:
  q_local=[2, 3, 5, 7]
  q_nope=[2, 3, 5, 7]
  q_rope=[]
  kv_a.out=[]
  kv_b.out=[2, 3, 5, 7, 258, 259, 261, 263, 514, 515, 517, 519, 770, 771, 773, 775, 1026, 1027, 1029, 1031, 1282, 1283, 1285, 1287, 1538, 1539, 1541, 1543, 1794, 1795, 1797, 1799, 2050, 2051, 2053, 2055, 2306, 2307, 2309, 2311, 2562, 2563, 2565, 2567, 2818, 2819, 2821, 2823, 3074, 3075, 3077, 3079, 3330, 3331, 3333, 3335, 3586, 3587, 3589, 3591, 3842, 3843, 3845, 3847]
  kv_b.in=[]
  o_proj.in=[]


In [65]:

debug_group_prune_step_by_step(correct_group)

LINEARS BEFORE {124172761230224: (2048, 3072, (3072, 2048), 124172745160192), 124172761231712: (2048, 2048, (2048, 2048), 124172741719712), 124172761228496: (2048, 576, (576, 2048), 124172741718432), 124172761233104: (512, 4096, (4096, 512), 124172745160512)}

[0] prune_out_channels target=blocks.0._original_component.self_attn._original_component.q_proj._original_component (Linear(in_features=2048, out_features=3072, bias=False)) idxs=64
CHANGED: (2048, 3072, (3072, 2048), 124172745160192) -> (2048, 3008, (3008, 2048), 124172744124400)

[1] prune_out_channels target=_Reshape_1581() idxs=64

[2] prune_out_channels target=_ElementWiseOp_1580(TransposeBackward0) idxs=64

[3] prune_out_channels target=_SplitOp_1572(None) idxs=64

[4] prune_out_channels target=_ConcatOp_1571(None) idxs=64

[5] prune_out_channels target=_ElementWiseOp_1579(ToCopyBackward0) idxs=64

[6] prune_out_channels target=_Reshape_1578() idxs=64

[7] prune_out_channels target=_Reshape_1577() idxs=64

[8] prune_out_cha

{124172761230224: (2048, 3008, (3008, 2048), 124172744124400),
 124172761231712: (2048, 2048, (2048, 2048), 124172741424160),
 124172761228496: (2048, 576, (576, 2048), 124172741420320),
 124172761233104: (512, 4032, (4032, 512), 124172741425360)}

In [68]:
correct_group, _ = gb.get_correct_pruning_group_and_structural_setup( bridge.blocks[0].mlp.up_proj, #not supported yet
    tp.prune_linear_out_channels,
    GroupPruneTask(None, [2, 3, 5, 7]))

AttributeError: 'Linear' object has no attribute 'hook_in'

In [ ]:
debug_group_prune_step_by_step(correct_group)

LINEARS BEFORE {124435119586704: (2560, 10240, (10240, 2560), 124435071110736), 124435119585072: (10240, 2560, (2560, 10240), 124435072836176), 124435119587184: (2560, 10240, (10240, 2560), 124435117785712)}

[0] prune_out_channels target=blocks.0._original_component.mlp._original_component.up_proj._original_component (Linear(in_features=2560, out_features=10240, bias=False)) idxs=4
CHANGED: (2560, 10240, (10240, 2560), 124435071110736) -> (2560, 10236, (10236, 2560), 124435082481472)

[1] prune_out_channels target=_ElementWiseOp_3660(MulBackward0) idxs=4

[2] prune_out_channels target=_ElementWiseOp_3661(GeluBackward0) idxs=4

[3] prune_out_channels target=_Reshape_3658() idxs=4

[4] prune_out_channels target=_ElementWiseOp_3657(MmBackward0) idxs=4

[5] prune_out_channels target=_ElementWiseOp_3659(TBackward0) idxs=4

[6] prune_in_channels target=blocks.0._original_component.mlp._original_component.down_proj._original_component (Linear(in_features=10240, out_features=2560, bias=False)

{124435119586704: (2560, 10236, (10236, 2560), 124435082481472),
 124435119585072: (10236, 2560, (2560, 10236), 124435082481232),
 124435119587184: (2560, 10236, (10236, 2560), 124435082483072)}

So deepseek works correctly after inner idxs fixing for kv_a, kv_b and other things (but not tested for q lora branch).  
But MOE undone (hook problems).  
MLP undone (problems with component that not bridge, cant attach hook in correct place).

Also problems, if we load quantized version in Linear4bit, that can not be with requires_grad=True and can not be pruned with TP for comparsion (but can be pruned with our hooks for activations)